# Chapter 18 &mdash; S and K, Actually Running

**Concept 11 of the Chapter 18 decomposition:** *S and K, Actually Running: Arithmetic and Recursion in SKI*

2 + 3 reaches $f(f(f(f(fx))))$ in 108 rewrites of three rules &mdash; and a recursion with a base case sums 5 down to 0 in 3,519.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter18-Lambda/Concept-SKI-Actually-Running/Concept-SKI-Actually-Running.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.AnimateSKI     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateSKI as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateSKI, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


Concept 10 ended with a claim: $S$ and $K$ suffice for everything. Here it runs.

Three rules, and nothing else &mdash; no numbers, no arithmetic, no machine:

$$I\,x \to x \qquad K\,x\,y \to x \qquad S\,x\,y\,z \to x\,z\,(y\,z)$$

**Bracket abstraction** turns any closed $\lambda$-term into those three, so the
Church numerals become SKI terms and arithmetic becomes application. `PLUS 2 3`
reduces in **108 steps** to

$$f(f(f(f(f\,x))))$$

which is not a representation of five &mdash; it *is* five, in the only sense Church
numerals have: $f$ applied five times.

`AnimateSKI()` walks those 108 steps with the **redex highlighted**, so a single
rewrite can be followed by eye: find the box, take a step, see what the box became.
$S$, $K$ and $I$ each keep their own colour, because which rule is firing is most of
what you want to know.

**Going further needs a conditional**, and there is one. A Church zero ignores its
first argument, so `ISZERO n a b` selects between $a$ and $b$ &mdash; that is an `if`.
With $Y$ to tie the knot, `sumto(5)` $= 5+4+3+2+1+0 = 15$ runs to completion.

**And then the bill arrives.** Those 3,519 steps pass through a term of
**1,145,178 symbols** &mdash; written out as a tree. Written as a **graph**, sharing
every repeated subterm, the same term is **374** symbols. Three thousand times
smaller, for the same computation. That gap is why no real implementation copies.

## 2. Definitions

### The three rules, and what they do to a term

In [ ]:
from jove.AnimateSKI import *

print('I x      ->', show(reduce_ski(app(I, 'x'))[0]))
print('K x y    ->', show(reduce_ski(app(K, 'x', 'y'))[0]))
print('S x y z  ->', show(reduce_ski(app(S, 'x', 'y', 'z'))[0]))
print()
print('S is the interesting one: it DUPLICATES z.  That single fact is')
print('why terms grow, why sharing matters, and why S is enough.')
print()
print('I is not even primitive:  SKK x ->', show(reduce_ski(app(S, K, K, 'x'))[0]))

### Church numerals, bracket-abstracted into SKI

In [ ]:
for n in range(4):
    t = encode(n)
    print('  %d  ->  %-46s  (%d symbols)' % (n, show(t), size(t)))
print()
print('decode() asks a numeral what it is the only way you can: give it a')
print('free f and a free x, reduce, and count the f you get back.')
for n in range(4):
    val, steps = decode(encode(n))
    print('  decode(encode(%d)) = %s   in %d steps' % (n, val, steps))

<!-- nav-strip -->

---

&larr;&nbsp;[Ch18&nbsp;10.&nbsp;Combinators, and the Universality of $S$ and $K$](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter18-Lambda/Concept-Combinators-S-And-K/Concept-Combinators-S-And-K.ipynb) &nbsp;&middot;&nbsp; [**Chapter 18** index](https://github.com/ganeshutah/Jove/blob/master/Chapter18-Lambda/README.md)

---

## 3. Tests

**2 + 3.** The whole of arithmetic here is application.

In [ ]:
val, steps = decode(app(PLUS, encode(2), encode(3)))
print('PLUS 2 3  =', val, ' in', steps, 'steps')

nf, _ = reduce_ski(app(PLUS, encode(2), encode(3), 'f', 'x'))
print('normal form:', show(nf))
assert val == 5 and show(nf) == 'f(f(f(f(fx))))'
print()
print('Five applications of a free f.  Nothing was counted; it was rewritten.')

It is not a fluke of 2 and 3.

In [ ]:
print('%3s %3s %5s %7s' % ('m', 'n', 'm+n', 'steps'))
for m in range(4):
    for n in range(4):
        v, st = decode(app(PLUS, encode(m), encode(n)))
        assert v == m + n
        if m + n in (0, 3, 5, 6):
            print('%3d %3d %5d %7d' % (m, n, v, st))
print()
print('All 16 checked.  MULT works too:')
for m, n in ((2, 3), (3, 3)):
    v, st = decode(app(MULT, encode(m), encode(n)))
    print('  %d x %d = %-3s in %5d steps' % (m, n, v, st))
    assert v == m * n

**A conditional**, which is what recursion needs. A Church zero ignores its first argument; every other numeral does not. That single asymmetry is an `if`.

In [ ]:
for n in range(4):
    got = show(reduce_ski(app(ISZERO, encode(n), 'then', 'else'))[0])
    print('  ISZERO %d then else  ->  %s' % (n, got))
print()
print('PRED is the expensive one -- it rebuilds the numeral one step behind:')
for n in range(4):
    v, st = decode(app(PRED, encode(n)))
    print('  pred %d = %s  in %3d steps' % (n, v, st))
assert decode(app(PRED, encode(3)))[0] == 2

**Recursion.** `sumto` is $Y$ applied to a function that tests for zero, stops, or adds $n$ to the answer for $n-1$. Nothing above this line was invented for it.

In [ ]:
print('%3s %7s %9s' % ('N', 'sumto', 'steps'))
for n in range(6):
    v, st = sumto(n)
    assert v == n * (n + 1) // 2
    print('%3d %7s %9d' % (n, v, st))
print()
print('sumto(5) = 15, in 3,519 rewrites of three rules.')
print('Checked against N(N+1)/2 for every N, so the SKI is not being')
print('graded by the SKI.')

**The bill.** The same reduction, measured two ways: the term as a tree of copies, and as a graph that shares. This is the argument for graph reduction, and it is not close.

In [ ]:
sharing_table(5)

## 4. Animation

Step through `PLUS 2 3 f x`. The highlight is the **redex** &mdash; the subterm
about to be rewritten &mdash; and the colour tells you which rule fires. Watch the
term balloon to 138 symbols and then collapse; the last six steps peel away the
scaffolding and five `f`s are simply there.

Try `AnimateSKI(app(SUCC, encode(2), 'f', 'x'), 'SUCC 2 f x')` for a shorter one, or
`AnimateSKI(app(MULT, encode(2), encode(2), 'f', 'x'), 'MULT 2 2 f x')`.

In [ ]:
from jove.AnimateSKI import *
AnimateSKI()

## 5. Exercises


1. `encode(1)` is ten symbols, not `I`. Bracket abstraction has no $\eta$ rule.
   Reduce `encode(1) f x` and `I f x` by hand and confirm they agree anyway.
2. Which rule fires most often in `PLUS 2 3` &mdash; $S$, $K$ or $I$? Count them
   from `trace()`, and say why that ordering is what you would expect.
3. The term peaks at 138 symbols and ends at 6. Plot size against step from
   `trace()`. Where is the peak, and what is happening there?
4. `fact(3)` is 6, in 5,558 steps. Try `fact(4)`. Time it before you run it, then
   explain the jump using `sharing_table(4, 'fact')`.
5. $I = SKK$. Replace every `I` in `encode(2)` with `SKK` and check the numeral
   still decodes to 2. How many more steps does it take?
6. `ISZERO` works because a Church zero ignores its first argument. Write out
   `ISZERO 0` and `ISZERO 1` as $\lambda$-terms and find the exact point where the
   two part company.
7. The tree/graph ratio for `sumto` grows $1, 12, 76, 333, 702, 3062$. Is that
   exponential in $N$? Fit it, and say what the growth means for an implementation
   that copies.

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for all 254 concepts.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter18-Lambda/Concept-SKI-Actually-Running')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')